# コーヒー2050年問題 ― 3カ国 × 将来予測分析
## エチオピア・コロンビア・グアテマラの適地が、どう変わるか

**Frontier Academy Fukuoka 卒業制作（2026年6月）**  
**作成者：** 森山光明（Mitsuaki Moriyama）

---

## 🎯 このノートブックの目的

前ノートブック（`ethiopia_coffee_suitability.ipynb`）でエチオピアの現状適地を可視化した。  
本ノートブックでは以下の3点に発展させる：

1. **3カ国への展開**：エチオピア・コロンビア・グアテマラ（スタバ主要調達国）
2. **将来予測**：気温+2.5℃シナリオ（IPCC SSP2-4.5 / 2050年中央値）
3. **スタバ調達への影響考察**：適地面積の変化を国別に定量化

## 📐 シナリオの根拠

IPCC AR6 (2021) によれば、SSP2-4.5シナリオ下での2041-2060年の世界平均気温上昇は約2.1〜2.7℃。本研究では**中央値の+2.5℃**を一律シフトとして適用する。

※ 厳密にはCMIP6の地域別ダウンスケーリングデータを用いるべきだが、卒業制作のスコープ上、一律シフト近似とする。これは Bunn et al. (2015) も類似アプローチを採用している。


---
## 1. 環境セットアップ

In [ ]:
!pip install japanize-matplotlib -q

import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib
from ipywidgets import HTML
import os

# 自分のGEEプロジェクトIDに置き換える
ee.Initialize(project='ethiopia-coffee-gee')

os.makedirs('outputs_2050', exist_ok=True)
print('✅ 環境セットアップ完了')

---
## 2. 国別パラメータ定義

3カ国の境界と主要産地を一括管理する。

In [ ]:
COUNTRIES = {
    'Ethiopia': {
        'gaul_name': 'Ethiopia',
        'center': [9.145, 40.4897],
        'zoom': 6,
        'regions': [
            {'name': 'Sidamo', 'lat': 6.77, 'lon': 38.50, 'desc': '柑橘系の華やかな酸味'},
            {'name': 'Yirgacheffe', 'lat': 6.16, 'lon': 38.20, 'desc': '花のような香り'},
            {'name': 'Harrar', 'lat': 9.31, 'lon': 42.12, 'desc': 'ワイニーで野性的'},
        ]
    },
    'Colombia': {
        'gaul_name': 'Colombia',
        'center': [4.5, -74.0],
        'zoom': 6,
        'regions': [
            {'name': 'Huila', 'lat': 2.53, 'lon': -75.52, 'desc': '世界最大の生産州・高品質'},
            {'name': 'Antioquia', 'lat': 6.25, 'lon': -75.57, 'desc': '伝統的なコーヒー文化'},
            {'name': 'Nariño', 'lat': 1.21, 'lon': -77.28, 'desc': '南部高地・スペシャルティ'},
        ]
    },
    'Guatemala': {
        'gaul_name': 'Guatemala',
        'center': [15.5, -90.25],
        'zoom': 7,
        'regions': [
            {'name': 'Antigua', 'lat': 14.55, 'lon': -90.73, 'desc': '世界的に有名な火山性土壌'},
            {'name': 'Huehuetenango', 'lat': 15.32, 'lon': -91.47, 'desc': '高品質スペシャルティ'},
            {'name': 'Atitlán', 'lat': 14.70, 'lon': -91.20, 'desc': '湖畔の独特な風味' },
        ]
    }
}

print('✅ 3カ国のパラメータ定義完了')

---
## 3. 共通関数の定義

国境取得、データ取得、適地スコア算出を関数化する。

In [ ]:
def get_country_geom(country_name):
    """国境ポリゴンを取得"""
    countries = ee.FeatureCollection('FAO/GAUL/2015/level0')
    country = countries.filter(ee.Filter.eq('ADM0_NAME', country_name))
    return country, country.geometry()

def get_climate_data(geom):
    """4種類の衛星データを取得"""
    # 標高
    elevation = ee.Image('USGS/SRTMGL1_003').clip(geom)
    
    # 気温（2020-2024平均）
    era5 = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR') \
        .filterDate('2020-01-01', '2025-01-01') \
        .select('temperature_2m')
    temp_c = era5.mean().subtract(273.15).clip(geom).rename('temp_c')
    
    # 年降水量
    chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY') \
        .filterDate('2020-01-01', '2025-01-01') \
        .select('precipitation')
    precip = chirps.mean().multiply(365).clip(geom).rename('precip_mm')
    
    # NDVI
    def mask_s2(image):
        qa = image.select('QA60')
        mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
        return image.updateMask(mask).divide(10000)
    
    s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterDate('2024-01-01', '2025-01-01') \
        .filterBounds(geom) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
        .map(mask_s2)
    ndvi = s2.median().normalizedDifference(['B8', 'B4']).clip(geom).rename('ndvi')
    
    return elevation, temp_c, precip, ndvi

def calculate_suitability(elevation, temp_c, precip, ndvi):
    """適地スコア算出（0〜1）"""
    elev_score = ee.Image(0) \
        .where(elevation.gte(1200).And(elevation.lt(1500)), 0.5) \
        .where(elevation.gte(1500).And(elevation.lte(2200)), 1.0) \
        .where(elevation.gt(2200).And(elevation.lte(2500)), 0.5)
    
    temp_score = ee.Image(0) \
        .where(temp_c.gte(15).And(temp_c.lt(18)), 0.5) \
        .where(temp_c.gte(18).And(temp_c.lte(22)), 1.0) \
        .where(temp_c.gt(22).And(temp_c.lte(24)), 0.5)
    
    precip_score = ee.Image(0) \
        .where(precip.gte(1000).And(precip.lt(1200)), 0.5) \
        .where(precip.gte(1200).And(precip.lte(1800)), 1.0) \
        .where(precip.gt(1800).And(precip.lte(2000)), 0.5)
    
    ndvi_score = ee.Image(0) \
        .where(ndvi.gte(0.3).And(ndvi.lt(0.4)), 0.5) \
        .where(ndvi.gte(0.4), 1.0)
    
    suitability = elev_score.multiply(0.30) \
        .add(temp_score.multiply(0.30)) \
        .add(precip_score.multiply(0.25)) \
        .add(ndvi_score.multiply(0.15)) \
        .rename('suitability')
    
    return suitability

print('✅ 共通関数定義完了')

---
## 4. 3カ国の現状＆2050年予測を一括計算

**2050年シナリオ：気温を一律 +2.5℃ シフト**

In [ ]:
TEMP_RISE_2050 = 2.5  # IPCC SSP2-4.5シナリオの中央値

results = {}

for country_name, params in COUNTRIES.items():
    print(f'⏳ {country_name} を計算中...')
    
    country, geom = get_country_geom(params['gaul_name'])
    elevation, temp_c, precip, ndvi = get_climate_data(geom)
    
    # 現状の適地スコア
    suitability_now = calculate_suitability(elevation, temp_c, precip, ndvi)
    
    # 2050年予測：気温を+2.5℃
    temp_2050 = temp_c.add(TEMP_RISE_2050)
    suitability_2050 = calculate_suitability(elevation, temp_2050, precip, ndvi)
    
    # 差分
    suitability_diff = suitability_2050.subtract(suitability_now).rename('diff')
    
    results[country_name] = {
        'country': country,
        'geom': geom,
        'elevation': elevation,
        'temp_now': temp_c,
        'temp_2050': temp_2050,
        'precip': precip,
        'ndvi': ndvi,
        'suitability_now': suitability_now,
        'suitability_2050': suitability_2050,
        'suitability_diff': suitability_diff,
        'params': params,
    }
    print(f'   ✅ {country_name} 計算完了')

print('\n✅ 全3カ国の現状＆2050予測計算完了')

---
## 5. 国別マップ可視化

各国について「現状」「2050年予測」「差分」の3レイヤーを表示する。

In [ ]:
suitability_vis = {
    'min': 0, 'max': 1,
    'palette': ['white', 'lightyellow', 'yellow', 'orange', 'red', 'darkred']
}
diff_vis = {
    'min': -0.7, 'max': 0.0,
    'palette': ['#5e0c1c', '#a63a26', '#e08e3c', '#f7d588', '#ffffff']
}

def make_country_map(country_name):
    """国別マップを作成（現状・2050・差分の3レイヤー）"""
    r = results[country_name]
    p = r['params']
    
    Map = geemap.Map(center=p['center'], zoom=p['zoom'])
    
    Map.addLayer(r['suitability_now'], suitability_vis, f'{country_name} 現状')
    Map.addLayer(r['suitability_2050'], suitability_vis, f'{country_name} 2050年(+2.5℃)', False)
    Map.addLayer(r['suitability_diff'], diff_vis, f'{country_name} 差分(2050-現状)', False)
    Map.addLayer(r['country'].style(color='black', fillColor='00000000', width=2), {}, '国境')
    
    # 主要産地ピン
    for region in p['regions']:
        popup = HTML(value=f"<b>{region['name']}</b><br>{region['desc']}")
        Map.add_marker(location=[region['lat'], region['lon']], popup=popup)
    
    return Map

# エチオピア
make_country_map('Ethiopia')

In [ ]:
# コロンビア
make_country_map('Colombia')

In [ ]:
# グアテマラ
make_country_map('Guatemala')

---
## 6. 主要産地スコアの検証（現状 vs 2050）

各国の主要産地で、現状と2050年予測のスコアを比較する。

In [ ]:
def safe_round(value, digits=2):
    if value is None:
        return None
    return round(value, digits)

all_region_results = []

for country_name, r in results.items():
    for region in r['params']['regions']:
        point = ee.Geometry.Point([region['lon'], region['lat']])
        buffer = point.buffer(5000)
        
        # 現状と2050のスコアを取得
        combined = ee.Image.cat([
            r['elevation'].rename('elev'),
            r['temp_now'],
            r['temp_2050'].rename('temp_2050'),
            r['precip'],
            r['ndvi'],
            r['suitability_now'].rename('score_now'),
            r['suitability_2050'].rename('score_2050'),
        ]).reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=buffer,
            scale=500,
            maxPixels=1e9
        ).getInfo()
        
        score_now = combined.get('score_now')
        score_2050 = combined.get('score_2050')
        diff = (score_2050 - score_now) if (score_now is not None and score_2050 is not None) else None
        
        all_region_results.append({
            '国': country_name,
            '産地': region['name'],
            '標高(m)': safe_round(combined.get('elev'), 0),
            '現気温(℃)': safe_round(combined.get('temp_c'), 1),
            '2050気温(℃)': safe_round(combined.get('temp_2050'), 1),
            '現スコア': safe_round(score_now, 2),
            '2050スコア': safe_round(score_2050, 2),
            '変化': safe_round(diff, 2),
        })

df_regions = pd.DataFrame(all_region_results)
print('=== 主要産地：現状 vs 2050年予測 ===')
df_regions

---
## 7. 国別の適地面積変化（定量化）

各国で「現状の良好＋最適面積」と「2050年の良好＋最適面積」を比較する。

In [ ]:
def calc_suitable_area(suitability, geom, threshold=0.5):
    """スコアthreshold以上の面積をkm²で算出"""
    pixel_area = ee.Image.pixelArea().divide(1e6)
    mask = suitability.gt(threshold)
    area = pixel_area.updateMask(mask).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=geom,
        scale=5000,
        maxPixels=1e10,
        tileScale=4
    ).getInfo()
    return area.get('area', 0)

area_results = []

for country_name, r in results.items():
    print(f'⏳ {country_name} の面積計算中...')
    area_now = calc_suitable_area(r['suitability_now'], r['geom'])
    area_2050 = calc_suitable_area(r['suitability_2050'], r['geom'])
    change_pct = ((area_2050 - area_now) / area_now * 100) if area_now > 0 else 0
    
    area_results.append({
        '国': country_name,
        '現状適地(km²)': round(area_now, 0),
        '2050適地(km²)': round(area_2050, 0),
        '消失面積(km²)': round(area_now - area_2050, 0),
        '変化率(%)': round(change_pct, 1),
    })
    print(f'   ✅ {country_name}: 現状 {area_now:,.0f} → 2050 {area_2050:,.0f} km² ({change_pct:+.1f}%)')

df_area = pd.DataFrame(area_results)
print('\n=== 国別：適地面積の変化（良好＋最適） ===')
df_area

---
## 8. 可視化：国別の影響を比較

In [ ]:
# 適地面積の変化（現状 vs 2050）を棒グラフで
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 左：面積の絶対値比較
ax1 = axes[0]
x = np.arange(len(df_area))
width = 0.35
bars1 = ax1.bar(x - width/2, df_area['現状適地(km²)'], width, label='現状', color='#2a9d8f', edgecolor='black')
bars2 = ax1.bar(x + width/2, df_area['2050適地(km²)'], width, label='2050年(+2.5℃)', color='#e76f51', edgecolor='black')
ax1.set_xticks(x)
ax1.set_xticklabels(df_area['国'], fontsize=12)
ax1.set_ylabel('適地面積 (km²)', fontsize=12)
ax1.set_title('国別：コーヒー適地面積の変化', fontsize=13, weight='bold')
ax1.legend(fontsize=11)
ax1.grid(axis='y', linestyle='--', alpha=0.4)
for bar in bars1 + bars2:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, h + max(df_area['現状適地(km²)']) * 0.01,
             f'{h:,.0f}', ha='center', fontsize=9)

# 右：変化率
ax2 = axes[1]
colors_change = ['#d62828' if v < 0 else '#2a9d8f' for v in df_area['変化率(%)']]
bars3 = ax2.barh(df_area['国'], df_area['変化率(%)'], color=colors_change, edgecolor='black')
ax2.axvline(0, color='black', linewidth=0.8)
ax2.set_xlabel('適地面積の変化率 (%)', fontsize=12)
ax2.set_title('2050年への変化率', fontsize=13, weight='bold')
ax2.grid(axis='x', linestyle='--', alpha=0.4)
for i, v in enumerate(df_area['変化率(%)']):
    ax2.text(v - 1.5 if v < 0 else v + 0.5, i, f'{v:+.1f}%',
             va='center', ha='right' if v < 0 else 'left', fontsize=11, weight='bold')

plt.tight_layout()
plt.savefig('outputs_2050/area_change_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ outputs_2050/area_change_comparison.png 保存完了')

In [ ]:
# 主要産地別のスコア変化
fig, ax = plt.subplots(figsize=(12, 7))

x = np.arange(len(df_regions))
width = 0.35

bars1 = ax.bar(x - width/2, df_regions['現スコア'], width, label='現状', color='#2a9d8f', edgecolor='black')
bars2 = ax.bar(x + width/2, df_regions['2050スコア'], width, label='2050年(+2.5℃)', color='#e76f51', edgecolor='black')

labels = [f"{row['産地']}\n({row['国'][:3]})" for _, row in df_regions.iterrows()]
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('適地スコア', fontsize=12)
ax.set_title('主要産地：現状 vs 2050年予測スコア', fontsize=14, weight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.set_ylim(0, 1.1)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.axhline(0.7, color='gray', linestyle=':', linewidth=1, alpha=0.7)
ax.text(8.5, 0.72, '最適ライン (0.7)', fontsize=9, color='gray')

for bar in bars1:
    h = bar.get_height()
    if h is not None:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.02, f'{h:.2f}', ha='center', fontsize=8)
for bar in bars2:
    h = bar.get_height()
    if h is not None:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.02, f'{h:.2f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('outputs_2050/region_score_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ outputs_2050/region_score_comparison.png 保存完了')

---
## 9. スタバ調達への影響考察

### 9-1. 国別影響サマリー

上記の数字を見ると、3カ国それぞれで2050年の影響度合いが異なる。

### 9-2. なぜ国によって違うのか

**標高分布の違い**が決定的：
- **エチオピア**：南西部に2000m級の高地が広く分布 → 気温上昇に対する「逃げ場」がある
- **コロンビア**：アンデス山脈に沿って高標高地帯あり → 同様に逃げ場あり
- **グアテマラ**：火山地帯で高地はあるが面積が限定的 → 影響が出やすい可能性

### 9-3. スタバの調達戦略への示唆

スターバックスはエシカル調達（C.A.F.E.プラクティス）を通じて、世界30カ国以上の農家と直接関係を築いている。本分析が示唆するのは：

1. **既存産地の維持コストが上昇**：気候適応のための農家支援が不可欠
2. **産地の地理的シフト**：より高標高への移動、新興産地の開拓
3. **品質と量のトレードオフ**：適地縮小により価格高騰の可能性

→ 衛星データによる**継続的モニタリングが調達リスク管理の核となる**ことを示している。

---
## 10. まとめ

### 達成したこと

1. ✅ エチオピア・コロンビア・グアテマラの3カ国へ適地モデルを展開
2. ✅ IPCC SSP2-4.5シナリオ（+2.5℃）による2050年予測マップを作成
3. ✅ 国別・産地別の適地スコア変化を定量化
4. ✅ スターバックス調達戦略への示唆を抽出

### 今後の発展

- CMIP6データによる地域別ダウンスケーリング
- 降水量変化の地域別シナリオ統合
- 機械学習による閾値モデルの精緻化
- ベトナム・ブラジル等の他主要産地への展開

---

*このノートブックは Frontier Academy Fukuoka 卒業制作の一部として、2026年6月に作成された。*